In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle
from sklearn.metrics import roc_auc_score


In [13]:
df_saudi_tree = pd.read_excel("saudi_dataset/Employee attrition dataset for tree-based models.xlsx")
df_saudi_nontree = pd.read_excel("saudi_dataset/Employee attrition dataset for non-tree-based models.xlsx")

In [14]:
df_saudi_tree.head()

,Attrition,Gender,Age,Maritalstatus,Academic_degree,Years_Experience,Years_experience_lastorganization,Sector,Department,JobTitle,...,Job_Engagement,Distance_to_work,Work_Live_Balance,Physical_Stress,Psychological_Exhaustion,Job_Stability,Health_Issues,Environment_Satisfaction,Job_Satisfaction,Job_Opportunities
0,1,0,1,0.405953,2,0,0,0.359278,0.600378,0.623277,...,1,1,1,2,0,0,1,1,0,1
1,0,0,0,0.447209,2,0,0,0.359278,0.600378,0.623277,...,1,1,1,0,0,1,0,1,1,0
2,0,0,0,0.447209,1,0,0,0.260830,0.229320,0.190450,...,1,0,1,2,1,0,0,1,1,1
3,0,0,0,0.447209,1,0,0,0.260830,0.229320,0.190450,...,1,1,1,2,2,0,0,0,0,0
4,0,0,0,0.447209,1,0,0,0.260830,0.229320,0.190450,...,1,1,1,1,1,0,0,1,0,0


In [15]:
columns_to_delete = ["Maritalstatus", "Department", "JobTitle", "Allowances"]
# allowances and marital status dropped since legend not clear 
# department and jobtitle dropped since we predict attrition for each sector, want enough data so we aggregate over the departments/jobs

df_saudi_tree = df_saudi_tree.drop(columns=columns_to_delete)

In [16]:
# Convert categorical string codes to integers
# 0.565187=1, 0.359278=2, 0.260830=3, 0.456738=4, 0.512493=5
sector_map = {0.5651866011647716: 1, 0.3592782301026135: 2, 0.2608303362988201: 3, 0.4567380973729042: 4, 0.5124929017603634: 5}

df_saudi_tree['Sector'] = df_saudi_tree['Sector'].map(sector_map)


In [6]:
df_saudi_tree.shape

(1174, 30)

In [7]:
# define protected attributes 
protected_attributes = ["Gender", "Age"]


In [17]:
random_state = 1234
saudi_train, saudi_test = train_test_split(df_saudi_tree, test_size=0.2, random_state=random_state)

saudi_train.to_parquet("saudi_dataset/train_cleaned.parquet")
saudi_test.to_parquet("saudi_dataset/test_cleaned.parquet")

# Separate features and target
x_train = saudi_train.drop("Attrition", axis=1)
y_train = saudi_train["Attrition"]
x_test = saudi_test.drop("Attrition", axis=1)
y_test = saudi_test["Attrition"]


In [18]:
# random forest model 
rf_model = RandomForestClassifier(n_estimators=100, random_state=random_state)
rf_model.fit(x_train, y_train)

#evaluate the model based on accuracy and AUC metrics 
rf_pred = rf_model.predict(x_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_pred)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")
print(f"AUC: {rf_auc * 100:.2f}%")


# Save the model
with open('saudi_dataset/RF.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

Accuracy: 80.85%
AUC: 79.19%


In [ ]:
feature_desc = ['the gender of the employee (0 = female, 1 = male)', 
                'the age of the employee (0 = 21-30, 1 = 31-40, 3= 41 and more)',
                'the highest obtained academic degree of the employee (0 = secondary school, 1 = bachelor, 2 = master, 3 = PhD)',
                'the total years of experience of the employee overall all their jobs (0 = 1-5 years, 1 = 6-10 years, 2 = 11 and more years)',
                'the years of experience the employee had in their previous job (0 = 1-5 years, 1 = 6-10 years, 2 = 11 and more years)',
                'the sector in which the employee is employed (1 = other , 2 = medical, 3 = education, 4 = financial, 5 = food)',
                'the monthly salary of the employee in SAR (0 = 1k-5k, 1 = 6k-10k, 2 = 11k-15k, 3 = 16k and more)',
                'whether the emloyee had medical ensurance in their last job (0 = no, 1 = yes)',
                'whether the employee received an annual bonus for their performance in their last job (0 = no, 1 = yes)',
                'whether the employee had overtime in their last job (0 = no, 1 = yes)', 
                'whether the employee received compenstation for their possible overtime in their last job (0 = no overtime, 1 = no, 2 = yes)',
                'whether the employee was satisfied with their income relative to their effort in their last job (0 = no, 1 = yes)',
                'whether the employee felt they got the promotion they deserved in their last job (0 = no, 1 = yes)',
                'the amount of training programs the employee attended in the last 3 years of their last job (0 = no trainings, 1 = 1-3, 2 = 4-6, 3 = 7 and more trainings)',
                'whether the employee benefited from the training programs provided by their former employer (0 = no, 1 = yes)',
                'how often the employee traveled for business purposes in their last job (0 = never, 1 = rarely, 2 = frequently)',
                'the level of support the employee received from their organization in their last job (0 = low, 1 = medium, 2 = high)',
                'whether the employee did feel feel the moral appreciation and recognition of their effort by their superiors in their previous job (0 = no, 1 = yes)',
                'the emotional commitment and psychological relationship of the employee with their former organization (0 = low, 1 = medium, 2 = high)', 
                'how easy it was to get involved in the job (e.g. decision making, opinions) of the employee in their last job (0 = easy, 1 = medium, 2 = difficult)', 
                'the distance to the employee his/her workplace in their last job (0 = close, 1 = medium, 2 = far)',
                'how easy it was to balance work life and personal life at work for the employee in their last job (0 = easy, 1 = medium, 2 = difficult)',
                'whether the employee felt physically stressed due to physically demanding tasks in their last job (0 = no, 1 = sometimes, 2 = yes)',
                'whether the employee felt a state of psychological exhaustion and mental or emotional fatigue as a result of their constant exposure to job stress in their last job (0 = no, 1 = sometimes, 2 = yes)',
                'whether the employee felt a sense of job security and stability in their last job (0 = no, 1 = yes)',
                'whether the employee did have any health issues that foreced them to leave their last job (0 = no, 1 = yes)',
                'the satisfaction of the employee with their work environment in their last job (0 = low, 1 = medium, 2 = high)',
                'the satisfaction of the employee with their previous job in general (0 = not satisfied, 1 = satisfied, 2 = very satisfied)',
                'whether the employee received other job opportunities while working in their last job (0 = no, 1 = yes)'
]

feature_desc_df = pd.DataFrame({
    "feature_name": list(x_test.columns),
    "feature_average": x_train.mean().to_list() ,
    "feature_desc": feature_desc,
})
     

dataset_description="data about the Saudi private sector to better understand the causes and consequences of employee satisfaction and turnover. The initial step to collect this data was administering an online survey to 1,200 employees of various Saudi Arabian companies. The dataset's question axes were described using a total of 29 qualities which you are given"
target_description="The target variable is whether the employee left their previous organization before their current job (1) or not (0)."
task_description="Predict whether a job applicant will leave the job they are applying for (1) or not (0) based on their characteristics."

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('saudi_dataset/dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [11]:
feature_desc_df

,feature_name,feature_average,feature_desc
0,Gender,0.424920,"the gender of the employee (0 = female, 1 = male)"
1,Age,0.624068,"the age of the employee (0 = 21-30, 1 = 31-40,..."
2,Academic_degree,0.963791,the highest obtained academic degree of the em...
3,Years_Experience,0.736954,the total years of experience of the employee ...
4,Years_experience_lastorganization,0.516507,the years of experience the employee had in th...
5,Sector,2.494143,the sector in which the employee is employed (...
6,MonthlySalary,0.798722,the monthly salary of the employee in SAR (0 =...
7,MedicalInsurance,0.708200,whether the emloyee had medical ensurance in t...
8,Bonus,0.422790,whether the employee received an annual bonus ...
9,OverTime,0.427050,whether the employee had overtime in their las...
